In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
hexes = ['#1D8348', '#5B2C6F', '#922B21', '#F1C40F']
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']

In [ ]:
clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna/csv/label_transfer/snmct_xgboost_v7.csv'
clusters = pd.read_csv(clusters_path, sep='\t', header=0, index_col=0)
clusters.shape

In [ ]:
var_dims_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/txt/dim_red/var_dims.txt'
var_dims = pd.read_csv(var_dims_path, header=None)[0].values
var_dims

In [ ]:
v = var_dims[0]
tmp_adata = sc.read_h5ad(f'{projdir}/h5ad/{v}/merged.processed.small.h5ad')
final_cells = tmp_adata.obs.index
final_cells.shape

In [ ]:
adatas = []
for v in var_dims:
    tmp_adata = sc.read_h5ad(f'{projdir}/h5ad/{v}/merged.processed.small.h5ad')
    tmp_adata.obs['cluster'] = clusters.reindex(tmp_adata.obs.index)['pred_coarse']
    tmp_adata.obs['donor_leiden'] = leiden.reindex(tmp_adata.obs.index)['donor_leiden']
    tmp_adata = tmp_adata[tmp_adata.obs.index.isin(final_cells)]
    adatas.append(tmp_adata)

In [ ]:
for v,tmp_adata in zip(var_dims, adatas):
    print(v)
    sc.pl.umap(tmp_adata, color=['donor', 'time_num', 'cluster', 'leiden'], ncols=2)

### per-donor

In [ ]:
# donor_adatas = []
for v in var_dims:
    print(v)
    for d in donors:
        tmp_adata = sc.read_h5ad(f'{projdir}/h5ad/{v}/{d}.processed.small.h5ad')
        tmp_adata.obs['cluster'] = clusters.reindex(tmp_adata.obs.index)['pred_coarse']
        tmp_adata.obs['donor_leiden'] = leiden.reindex(tmp_adata.obs.index)['donor_leiden']
        tmp_adata = tmp_adata[tmp_adata.obs.index.isin(final_cells)]
        sc.pl.umap(tmp_adata, color=['donor', 'time_num', 'cluster', 'donor_leiden'], ncols=2)
#         donor_adatas.append(tmp_adata)

### write out

In [ ]:
df_list = []
for v in var_dims:
    tmp_adata = sc.read_h5ad(f'{projdir}/h5ad/{v}/merged.processed.small.h5ad')
    tmp_adata.obs['cluster'] = clusters.reindex(tmp_adata.obs.index)['pred_coarse']
    tmp_adata.obs['donor_leiden'] = leiden.reindex(tmp_adata.obs.index)['donor_leiden']
    tmp_adata = tmp_adata[tmp_adata.obs.index.isin(final_cells)]
    umap_df = pd.DataFrame(index = tmp_adata.obs.index)
    umap_df['umap0'] = np.array(tmp_adata.obsm['X_umap'][:,0])
    umap_df['umap1'] = np.array(tmp_adata.obsm['X_umap'][:,1])
    umap_df['cluster'] = tmp_adata.obs['cluster'].copy()
    umap_df['donor_leiden'] = tmp_adata.obs['donor_leiden'].copy()
    umap_df['donor'] = tmp_adata.obs['donor'].copy()
    umap_df['time_num'] = tmp_adata.obs['time_num'].copy()
    umap_df['var_dim'] = v
    df_list.append(umap_df)

In [ ]:
merged = pd.concat(df_list)
merged.head()

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure2'
merged.to_csv(f'{outdir}/snmct_mcg_feature_umap.csv', sep='\t')

In [ ]:
!date